# Quick Start: Future Data Collection (READ THIS)

### **Scenario: You are back in the lab for a new recording session.**
If you have already trained the models and the cameras are in **roughly the same position** as last time, do not re-do the whole process.

**Follow these exact steps:**

1. **Record your new videos:**
   - After the cameras are in position, turn the lights off for 3-5 seconds then back on.
   - At any time before removing the cameras wave the Wand in various positions around the tank in such a way that it remains visible to all cameras. This can be done at the beginning, end, middle, does not matter. 
   - Film the Sharks, perform the experiment.
   - Use losslesscut to trim the videos so they all start at the light flicker sync moment.

2. **Update the Paths:**
   - Scroll down to **Step 4: Analysis & Anipose Prep**.
   - Update the `video_map` with the paths to your **NEW** video files.

3. **Run Specific Blocks Only:**
   - Run **Part 0** (Dependencies) & **Part 1** (Imports/Config).
   - **SKIP** Step 2 (Labeling) and Step 3 (Training).
   - **RUN** Step 4 (Analysis).
   - **RUN** Step 5 (Calibration).

**That's it!** The system will use your existing "Universal Wand Model" to recalibrate for today's camera positions.

# Marine Tracking Pipeline: 4-Camera Setup (v7 - Dynamic Wand)
**Goal:** Track Sandbar Sharks across 4 cameras (Top, Left, Right, Front) and automate 3D calibration.
- If you are building new models from nothing, follow all instructions. 
- If you already have dependencies installed and a development environment, skip to step 1.
- If you already have trained models, follow the instructions above. 

## 📦 Part 0: Dependencies & Setup (For New Computers)
**Prerequisite:** Download and Install [Anaconda (Miniconda)](https://docs.conda.io/en/latest/miniconda.html) for Windows.

If this is a fresh computer, open **Anaconda Prompt (Run as Administrator)** and run these blocks one by one to set up the environment.

If you have a preferred CLI, feel free to use it, but using a different CLI will change some of the steps to follow. 

You will need some version of a Python Notebook development environment to run the code blocks. VSCode with the Jupyter Notebook extension works great.

The following lines need to be run in Anaconda Prompt to build the development environment and install required dependencies the first time. Every other time, you simply need to run "conda activate dlc" to ensure your computer has all the necessary packages loaded. 

### 1. Create Environment & Install GPU Drivers
This installs the specific drivers (CUDA/cuDNN) needed for the graphics card to train the AI.
```powershell
# Create a clean environment named 'dlc'
conda create -n dlc python=3.8 -y
conda activate dlc

# Install GPU Support (Required for fast training)
conda install -c conda-forge cudatoolkit=11.2 cudnn=8.1.0 -y
```

### 2. Install DeepLabCut & Anipose
This installs the main software (DeepLabCut handles TensorFlow/QT automatically).
```powershell
# Install DeepLabCut with GUI support
pip install "deeplabcut[tf,gui]"

# Install the 3D Calibration Tool
pip install anipose
```

### 3. Fix Compatibility Issues (Crucial)
Anipose installs a new version of NumPy that breaks Napari. We must downgrade it manually.
```powershell
pip install "numpy<2.0"
pip install "napari<0.5.0"
```

## Part 1: Project Setup (The GUI)

### How to use the block below:
Running this block opens a **Configuration Window**.
You will need to tell it:
1. **Project Folder:** Where your deeplabcut project folders live, or where you want them to live if they don't exist yet. It checks if they exist first, then makes the necessary folders if they do not. 
2. **Video Folder:** Where today's `.mp4` files are. Cut the videos first and place them where you want them to live for the duration of the project. Moving them will be problematic. 
3. **Wand Start/End Time:** Look at your video timestamps. When does the calibration wand waving start and stop? (e.g. `0:30` to `1:45`).
4. **FPS:** Frame rate of the video (usually 30 or 60). Please syncronize all of the videos. If one is 30, they should all be 30.
5. **Wand Geometry:** Enter the Z (height) and X (width) coordinates of your wand markers in millimeters. The defaults match the 'Zig-Zag' wand.

In [ ]:
# --- INTERACTIVE PROJECT SETUP (v7.4 - Path Independent) ---
import deeplabcut
import yaml
import os
import shutil
import toml
import numpy as np
import pandas as pd
from pathlib import Path
import tkinter as tk
from tkinter import filedialog, messagebox

# --- DYNAMIC PATH DETECTION ---
# If script is in /Scripts, the Project Root is one level up
SCRIPT_DIR = os.getcwd()
DEFAULT_ROOT = str(Path(SCRIPT_DIR).parent) if "Scripts" in SCRIPT_DIR else SCRIPT_DIR

# --- GUI FUNCTIONS ---
def browse_folder(entry_field, title):
    folder = filedialog.askdirectory(initialdir=DEFAULT_ROOT, title=title)
    if folder:
        entry_field.delete(0, tk.END)
        entry_field.insert(0, folder)

def submit():
    global WORKING_DIR, VIDEO_DIR, WAND_START_FRAME, WAND_END_FRAME, FPS, WAND_Z, WAND_X
    WORKING_DIR = entry_proj.get()
    VIDEO_DIR = entry_vid.get()
    start_str = entry_start.get()
    end_str = entry_end.get()
    fps_str = entry_fps.get()
    z_str = entry_z.get()
    x_str = entry_x.get()

    if not WORKING_DIR or not VIDEO_DIR:
        messagebox.showerror("Error", "Please select both folders.")
        return
    
    try:
        FPS = int(fps_str)
        
        def time_to_frames(t_str, fps):
            if ':' in t_str:
                m, s = map(int, t_str.split(':'))
                seconds = m * 60 + s
            else:
                seconds = int(t_str)
            return seconds * fps

        WAND_START_FRAME = time_to_frames(start_str, FPS)
        WAND_END_FRAME = time_to_frames(end_str, FPS)

        WAND_Z = [float(i.strip()) for i in z_str.split(',')]
        WAND_X = [float(i.strip()) for i in x_str.split(',')]
        
        if len(WAND_Z) != len(WAND_X) or len(WAND_Z) < 2:
             messagebox.showerror("Error", "Wand Z and X must have the same number of points (at least 2).")
             return

        root.destroy()
    except ValueError:
        messagebox.showerror("Error", "Invalid Input. Check your Time formats, FPS, and Wand Coordinates.")

# --- BUILD WINDOW ---
root = tk.Tk()
root.title("Shark Tracking Setup")
root.attributes('-topmost', True)

tk.Label(root, text="Project Folder (D:\\...\\fish-dlc):").grid(row=0, column=0, sticky='w', padx=10, pady=5)
entry_proj = tk.Entry(root, width=50)
entry_proj.grid(row=0, column=1, padx=10, pady=5)
tk.Button(root, text="Browse", command=lambda: browse_folder(entry_proj, "Select Project Folder")).grid(row=0, column=2, padx=10)

tk.Label(root, text="Video Folder (D:\\...\\Michael):").grid(row=1, column=0, sticky='w', padx=10, pady=5)
entry_vid = tk.Entry(root, width=50)
entry_vid.grid(row=1, column=1, padx=10, pady=5)
tk.Button(root, text="Browse", command=lambda: browse_folder(entry_vid, "Select Video Folder")).grid(row=1, column=2, padx=10)

tk.Label(root, text="Wand Start Time (MM:SS):", fg="blue").grid(row=2, column=0, sticky='w', padx=10, pady=5)
entry_start = tk.Entry(root, width=10)
entry_start.insert(0, "0:19")
entry_start.grid(row=2, column=1, sticky='w', padx=10)

tk.Label(root, text="Wand End Time (MM:SS):", fg="blue").grid(row=3, column=0, sticky='w', padx=10, pady=5)
entry_end = tk.Entry(root, width=10)
entry_end.insert(0, "2:25")
entry_end.grid(row=3, column=1, sticky='w', padx=10)

tk.Label(root, text="FPS:", fg="green").grid(row=4, column=0, sticky='w', padx=10, pady=5)
entry_fps = tk.Entry(root, width=10)
entry_fps.insert(0, "30")
entry_fps.grid(row=4, column=1, sticky='w', padx=10)

tk.Label(root, text="Wand Z-Coords (mm):", fg="purple").grid(row=5, column=0, sticky='w', padx=10, pady=5)
entry_z = tk.Entry(root, width=50)
entry_z.insert(0, "43, 190, 372, 597, 875")
entry_z.grid(row=5, column=1, padx=10)

tk.Label(root, text="Wand X-Coords (mm):", fg="purple").grid(row=6, column=0, sticky='w', padx=10, pady=5)
entry_x = tk.Entry(root, width=50)
entry_x.insert(0, "100, -100, 100, -100, 100")
entry_x.grid(row=6, column=1, padx=10)

tk.Button(root, text="START PIPELINE", bg="green", fg="white", command=submit).grid(row=7, column=1, pady=20)

print("Waiting for GUI input...")
root.mainloop()

print(f"\n✅ Settings Locked In:")
print(f"   - Projects: {WORKING_DIR}")
print(f"   - Videos: {VIDEO_DIR}")

# --- AUTO-DETECT / AUTO-CREATE PROJECTS ---
project_configs = {}
found_folders = [f.path for f in os.scandir(WORKING_DIR) if f.is_dir()]
for p_path in found_folders:
    cfg_path = os.path.join(p_path, "config.yaml")
    if os.path.exists(cfg_path):
        fname = os.path.basename(p_path)
        if "Wand" in fname: project_configs['Wand'] = cfg_path
        elif "Top" in fname: project_configs['Top'] = cfg_path
        elif "Left" in fname: project_configs['Left'] = cfg_path
        elif "Right" in fname: project_configs['Right'] = cfg_path
        elif "Front" in fname: project_configs['Front'] = cfg_path

REQUIRED_VIEWS = ['Wand', 'Top', 'Left', 'Right', 'Front']
video_files = [os.path.join(VIDEO_DIR, f) for f in os.listdir(VIDEO_DIR) if f.lower().endswith('.mp4')]

for view in REQUIRED_VIEWS:
    if view not in project_configs:
        print(f"\n⚠️ '{view}' project missing. Creating it now...")
        
        # --- SIMPLE MATCHING LOGIC ---
        if view == 'Wand':   search_terms = ['tp', 'top'] 
        elif view == 'Top':  search_terms = ['tp', 'top']
        elif view == 'Left': search_terms = ['w1', 'left']
        elif view == 'Right':search_terms = ['w2', 'right']
        elif view == 'Front':search_terms = ['w3', 'front']

        match = next((v for v in video_files if any(s in os.path.basename(v).lower() for s in search_terms)), None)
        
        if match:
            try:
                print(f"   🎥 Using video: {os.path.basename(match)}")
                proj_name = f"Calibration_Wand" if view == 'Wand' else f"SandbarShark_{view}"
                # CHANGED: copy_videos=False
                cfg_path = deeplabcut.create_new_project(
                    proj_name, "Michael", [match], 
                    working_directory=WORKING_DIR, 
                    copy_videos=False, 
                    multianimal=False
                )
                project_configs[view] = cfg_path
            except Exception as e: print(f"   ❌ Error creating {view}: {e}")
        else: 
            print(f"   ❌ ERROR: Could not find video for '{view}'. Checked for: {search_terms}")

if len(project_configs) < 5: print("\n⚠️ WARNING: Missing projects.")
else: print("\n✅ System Ready.")

Loading DLC 2.3.9...


c:\Users\bhvlab\.conda\envs\dlc\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Waiting for GUI input...

✅ Settings Locked In:
   - Projects: D:/Data/SER/Edited/2026pilot/Michael/fish-dlc
   - Videos: D:/Data/SER/Edited/2026pilot/Michael

✅ System Ready.


In [2]:
# --- SETUP: CONFIGURE 3D GEOMETRY ---
shark_parts = ['Snout', 'DorsalFin', 'LeftPectoral', 'RightPectoral', 'TailBase', 'TailTip']
shark_skeleton = [['Snout', 'DorsalFin'], ['DorsalFin', 'TailBase'], ['TailBase', 'TailTip'], 
                  ['LeftPectoral', 'DorsalFin'], ['RightPectoral', 'DorsalFin']]
# Generate Bodyparts list based on input length (Ball1, Ball2... BallN)
wand_parts = [f'Ball{i+1}' for i in range(len(WAND_Z))]
wand_skeleton = [[wand_parts[i], wand_parts[i+1]] for i in range(len(wand_parts)-1)]

for name, cfg_path in project_configs.items():
    with open(cfg_path, 'r') as f: cfg = yaml.safe_load(f)
    if name == 'Wand':
        cfg['bodyparts'] = wand_parts
        cfg['skeleton'] = wand_skeleton
        cfg['pcutoff'] = 0.9
    else:
        cfg['bodyparts'] = shark_parts
        cfg['skeleton'] = shark_skeleton
        cfg['pcutoff'] = 0.8
    with open(cfg_path, 'w') as f: yaml.safe_dump(cfg, f)

# --- GENERATE ANIPOSE CONFIG ---
ball_z_mm = np.array(WAND_Z, dtype=np.float32)
ball_x_mm = np.array(WAND_X, dtype=np.float32)

constraints = []
# Sequential Connections
for i in range(len(ball_z_mm) - 1): 
    dist = np.sqrt((ball_x_mm[i] - ball_x_mm[i+1])**2 + (ball_z_mm[i] - ball_z_mm[i+1])**2)
    constraints.append( (f"Ball{i+1}-Ball{i+2}", dist) )
# Skip Connections (Rigidity)
for i in range(len(ball_z_mm) - 2): 
    dist = np.sqrt((ball_x_mm[i] - ball_x_mm[i+2])**2 + (ball_z_mm[i] - ball_z_mm[i+2])**2)
    constraints.append( (f"Ball{i+1}-Ball{i+3}", dist) )

anipose_config = {
    "project": "Shark_Tracking_Metric",
    "model_folder": WORKING_DIR,
    "video_extension": "MP4", 
    "calibration": {
        "calibration_type": "wand",
        "animal_calibration": False,
        "fisheye": False
    },
    "triangulation": {
        "cam_regex": "Cam_([A-Za-z0-9]+)", 
        "optim": True,
        "constraints": [[p[0].split('-')[0], p[0].split('-')[1], float(p[1])] for p in constraints],
        "scale_smooth": 2,
        "spatial_smooth": 2,
        "repro_error_threshold": 15
    },
    "labeling": {
        "scheme": [wand_parts]
    }
}
with open(os.path.join(WORKING_DIR, "config.toml"), "w") as f: toml.dump(anipose_config, f)
print(f"✅ Project Geometry Updated for {len(WAND_Z)} points.")

✅ Project Geometry Updated for 5 points.


In [ ]:
# --- STEP 1.5: ROBUST FRAME EXTRACTION (Time-Gated) ---
import cv2
import numpy as np
import os
import yaml
import deeplabcut
from glob import glob

BASE_KMEANS = 20  # Frames for Top view

# Ensure GUI variables exist
if 'WAND_START_FRAME' not in globals():
    raise ValueError("Wand Start/End times not defined. Please run Step 4 (GUI) first.")

print(f"⏱️ Extraction Window for Wand: Frame {WAND_START_FRAME} to {WAND_END_FRAME}")

# --- FUNCTION: CUSTOM OPENCV EXTRACTOR ---
def custom_extract_wand(cfg_path, video_list, start_frame, end_frame, num_frames=40):
    """
    Manually extracts frames from a specific time window to avoid empty video segments.
    """
    # Load config to find the project path
    with open(cfg_path, 'r') as f:
        cfg = yaml.safe_load(f)
        
    for video_path in video_list:
        # Get video name without extension (e.g., 'myvideo.mp4' -> 'myvideo')
        vid_name = os.path.splitext(os.path.basename(video_path))[0]
        
        # Path: project/labeled-data/myvideo/
        output_dir = os.path.join(os.path.dirname(cfg_path), 'labeled-data', vid_name)
        os.makedirs(output_dir, exist_ok=True)
        
        print(f"   🎥 Processing: {vid_name}")
        cap = cv2.VideoCapture(video_path)
        
        if not cap.isOpened():
            print(f"      ❌ Error opening video: {video_path}")
            continue

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        # Safety Check: Ensure our window fits in the video
        real_end = min(end_frame, total_frames - 1)
        if start_frame >= real_end:
            print(f"      ⚠️ Window {start_frame}-{real_end} is invalid for this video (Length: {total_frames}). Skipping.")
            continue

        # Pick 40 frames evenly spaced inside your window
        indices = np.linspace(start_frame, real_end, num_frames, dtype=int)
        
        saved_count = 0
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                # Image Name Format: img01234.png (Must match frame number!)
                img_name = f"img{int(idx):05d}.png"
                cv2.imwrite(os.path.join(output_dir, img_name), frame)
                saved_count += 1
        
        cap.release()
        print(f"      ✅ Extracted {saved_count} frames (Range: {indices[0]}-{indices[-1]})")


# --- MAIN EXTRACTION LOOP ---
for name, cfg_path in project_configs.items():
    print(f"\n📸 EXTRACTING: {name}")

    # 1. SPECIAL HANDLING FOR WAND (Time-Gated + All Videos)
    if name == 'Wand':
        # Find ALL videos (Top, Left, Right, Front)
        all_videos = [os.path.join(VIDEO_DIR, f) for f in os.listdir(VIDEO_DIR) if f.lower().endswith('.mp4')]
        
        # Filter for videos that match our keywords
        wand_targets = [v for v in all_videos if any(k in os.path.basename(v).lower() for k in ['tp', 'top', 'w1', 'left', 'w2', 'right', 'w3', 'front'])]
        
        if wand_targets:
            print(f"   🔗 Linking {len(wand_targets)} videos to Wand Project...")
            deeplabcut.add_new_videos(cfg_path, wand_targets, copy_videos=False)
            
            # Run our Custom Extractor instead of DLC's default
            custom_extract_wand(cfg_path, wand_targets, WAND_START_FRAME, WAND_END_FRAME, num_frames=40)
        else:
            print("   ⚠️ No videos found for Wand extraction.")

    # 2. STANDARD HANDLING FOR SHARKS (Whole Video is fine)
    else:
        # Define rules per camera
        if name == 'Top':
            count = BASE_KMEANS
            algorithm = 'kmeans'
        elif name == 'Front':
            count = min(100, 4 * BASE_KMEANS)
            algorithm = 'uniform'
        else: # Left/Right
            count = min(100, 3 * BASE_KMEANS)
            algorithm = 'uniform'

        print(f"   - Algo: {algorithm}, Count: {count}")

        # Update Config
        with open(cfg_path, 'r') as f: cfg = yaml.safe_load(f)
        cfg['numframes2pick'] = count
        with open(cfg_path, 'w') as f: yaml.safe_dump(cfg, f)

        # Run Standard DLC Extraction
        try:
            deeplabcut.extract_frames(cfg_path, mode='automatic', algo=algorithm, userfeedback=False, crop=False)
            print("   ✅ Standard Extraction Done.")
        except Exception as e:
            print(f"   ⚠️ Extraction Skipped: {e}")

# --- STEP 2.5: STORAGE CLEANUP ---
import os
import glob

print("Scrubbing duplicated video files from project folders to save space...")

# Assuming 'project_configs' is your dictionary of paths from Step 1
for cam_name, cfg_path in project_configs.items():
    proj_dir = os.path.dirname(cfg_path)
    vid_dir = os.path.join(proj_dir, "videos")
    
    if os.path.exists(vid_dir):
        # Find all MP4s inside the project's video folder
        copied_vids = glob.glob(os.path.join(vid_dir, "*.mp4"))
        
        for v in copied_vids:
            try:
                os.remove(v)
                print(f"   🗑️ Deleted heavy copy: {os.path.basename(v)}")
            except Exception as e:
                print(f"   ❌ Could not delete {v}: {e}")

print("✅ Cleanup complete. Project folders are lightweight again.")

⏱️ Extraction Window for Wand: Frame 570 to 4350

📸 EXTRACTING: Wand
   🔗 Linking 4 videos to Wand Project...
Attempting to create a symbolic link of the video ...
Video D:\Data\SER\Edited\2026pilot\Michael\fish-dlc\Calibration_Wand-Michael-2026-02-17\videos\20260121_SERt2tp_goproGX010004_trimmed.mp4 already exists. Skipping...
Symlink creation impossible (exFat architecture?): copying the video instead.
D:\Data\SER\Edited\2026pilot\Michael\20260121_SERt2w1_goproGX010125_trimmed.mp4 copied to D:\Data\SER\Edited\2026pilot\Michael\fish-dlc\Calibration_Wand-Michael-2026-02-17\videos\20260121_SERt2w1_goproGX010125_trimmed.mp4
Symlink creation impossible (exFat architecture?): copying the video instead.
D:\Data\SER\Edited\2026pilot\Michael\20260121_SERt2w2_goproGX010153_trimmed.mp4 copied to D:\Data\SER\Edited\2026pilot\Michael\fish-dlc\Calibration_Wand-Michael-2026-02-17\videos\20260121_SERt2w2_goproGX010153_trimmed.mp4
Symlink creation impossible (exFat architecture?): copying the video i

9032it [06:21, 23.69it/s]
c:\Users\bhvlab\.conda\envs\dlc\lib\site-packages\sklearn\cluster\_kmeans.py:1934: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=3)
c:\Users\bhvlab\.conda\envs\dlc\lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\bhvlab\.conda\envs\dlc\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "c:\Users\bhvlab\.conda\envs\dlc\lib\subprocess.py", line 493, in run
    with Popen(*popenargs, **kwargs) as process:
  Fi

Kmeans clustering ... (this might take a while)
Frames were successfully extracted, for the videos listed in the config.yaml file.

You can now label the frames using the function 'label_frames' (Note, you should label frames extracted from diverse videos (and many videos; we do not recommend training on single videos!)).
   ✅ Standard Extraction Done.

📸 EXTRACTING: Left
   - Algo: uniform, Count: 60
Config file read successfully.
Extracting frames based on uniform ...
Uniformly extracting of frames from 0.0  seconds to 309.77  seconds.
Frames were successfully extracted, for the videos listed in the config.yaml file.

You can now label the frames using the function 'label_frames' (Note, you should label frames extracted from diverse videos (and many videos; we do not recommend training on single videos!)).
   ✅ Standard Extraction Done.

📸 EXTRACTING: Right
   - Algo: uniform, Count: 60
Config file read successfully.
Extracting frames based on uniform ...
Uniformly extracting of fram

## 🏷️ Step 2: Labeling (SKIP IF TRAINED)
**Use this only if you need to retrain the model.**
**This is not a code block, but it is manual labor that needs to be done in order to train your new model. Do not skip on new models.**

In order to label the frames, you will need to go outside this script and use a program called napari. 

Open the powershell, activate the python environment if applicable, then run ```napari```

Within napari:

1. Select Plugins > Keypoint controls (napari-deeplabcut)
2. Select File > Open File(s) and navigate to the config.yaml inside the project folder for the model you plan to label.
3. Select File > Open Folder and navigate to the folder containing the extracted frames within the same project. This is in the labeled-data/videoname folder by default

Napari should load controls for labeling and the extracted frames that need to be labeled. 

Relevant Controls:

On the right side panel, you can select which bodypart you are labeling. 

On the left panel there are some options to change the colors and size of the nodes. It's a good idea to play with it until you're comfortable. It's useful to associate a color with a bodypart in case you accidentally mislabel or get them out of order, it makes the mistake obvious to fix.

On the top left, there are navigation controls. 1. Delete, 2. Place node, 3. Select, 4. Move. 

Do not force it with the labeling. Skip frames that do not have the animal, and if a bodypart is not visible, do not label it. The model needs to learn both what the animal looks like, and what the view looks like when the animal is *not* there, or certain bodyparts are *not* visible. 

It is best to save and close, then reopen napari in between labeling each folder. It can get buggy if you try to jump between labeling sets. 

## 🏋️ Step 3: Training
This step will loop through all of the projects you created and train the model using the data you labeled. 

**Advanced**
It includes a safe-mode patch that downsamples video by 4x for two reasons. 
1. Deeplabcut does not support 4k video, but we want to use 4k video because it's awesome. So we cheat and lie to deeplabcut. 
2. Unless you're using a hefty lab computer with massive VRAM overhead, your computer isn't going to enjoy this step. The downsampling allows it to complete without crashing. 
If you aren't using 4k, or are using a hefty expensive lab machine, feel free to edit or remove the safe mode patch. Otherwise, just leave it, it's fine. 

**Warning**
This step takes a long time and your computer will get hot. Be sure to stick around for a minute and read the block's output to ensure there aren't any errors. When it starts printing something that looks like the following, you're free to walk away and let it run overnight:
Starting training....
iteration: 100 loss: 0.0609 lr: 0.005

In [2]:
# --- STEP 3: TRAINING (SKIP IF TRAINED) ---

def apply_safe_mode_patch(cfg_path):
    proj_dir = Path(cfg_path).parent
    found_files = list(proj_dir.glob("dlc-models/iteration-0/*/train/pose_cfg.yaml"))
    if found_files:
        pose_cfg_path = found_files[0]
        with open(pose_cfg_path, 'r') as f: train_cfg = yaml.safe_load(f)
        train_cfg['max_input_size'] = 4000
        train_cfg['global_scale'] = 0.25
        train_cfg['batch_size'] = 1 
        with open(pose_cfg_path, 'w') as f: yaml.safe_dump(train_cfg, f)
        print(f"   ✅ Applied Safe Mode to {proj_dir.name}")

# Run Loop
for name, cfg in project_configs.items():
    print(f"\n🚀 PROCESSING: {name}")
    try: deeplabcut.create_training_dataset(cfg, net_type='resnet_50', augmenter_type='imgaug')
    except: pass 
    apply_safe_mode_patch(cfg)
    
    print(f"   🧠 Training...")
    iters = 10000 if name == 'Wand' else 30000
    try: deeplabcut.train_network(cfg, shuffle=1, displayiters=200, saveiters=5000, maxiters=iters, allow_growth=True)
    except KeyboardInterrupt: print(f"   User skipped {name}...")
    except Exception as e: print(f"   ❌ Training failed: {e}")

print("\n✅ All Training Loops Completed.")
import time

print("Hibernating in 60 seconds, press ctrl+C to cancel.")
time.sleep(60)
os.system("shutdown /h")



🚀 PROCESSING: Wand


Config:
{'all_joints': [[0], [1], [2], [3], [4]],
 'all_joints_names': ['Ball1', 'Ball2', 'Ball3', 'Ball4', 'Ball5'],
 'alpha_r': 0.02,
 'apply_prob': 0.5,
 'batch_size': 1,
 'contrast': {'clahe': True,
              'claheratio': 0.1,
              'histeq': True,
              'histeqratio': 0.1},
 'convolution': {'edge': False,
                 'emboss': {'alpha': [0.0, 1.0], 'strength': [0.5, 1.5]},
                 'embossratio': 0.1,
                 'sharpen': False,
                 'sharpenratio': 0.3},
 'crop_pad': 0,
 'cropratio': 0.4,
 'dataset': 'training-datasets\\iteration-0\\UnaugmentedDataSet_Calibration_WandFeb17\\Calibration_Wand_Michael95shuffle1.mat',
 'dataset_type': 'imgaug',
 'decay_steps': 30000,
 'deterministic': False,
 'display_iters': 1000,
 'fg_fraction': 0.25,
 'global_scale': 0.25,
 'init_weights': 'c:\\Users\\bhvlab\\.conda\\envs\\dlc\\lib\\site-packages\\deeplabcut\\pose_estimation_tensorflow\\models\\pretrained\\resnet_v1_50.ckpt',
 'intermediate_supe

The training dataset is successfully created. Use the function 'train_network' to start training. Happy training!
   ✅ Applied Safe Mode to Calibration_Wand-Michael-2026-02-17
   🧠 Training...
Selecting single-animal trainer
Batch Size is 1


c:\Users\bhvlab\.conda\envs\dlc\lib\site-packages\tensorflow\python\keras\engine\base_layer_v1.py:1694: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  warnings.warn('`layer.apply` is deprecated and '


Loading ImageNet-pretrained resnet_50
Max_iters overwritten as 10000
Display_iters overwritten as 200
Save_iters overwritten as 5000
Training parameter:
{'stride': 8.0, 'weigh_part_predictions': False, 'weigh_negatives': False, 'fg_fraction': 0.25, 'mean_pixel': [123.68, 116.779, 103.939], 'shuffle': True, 'snapshot_prefix': 'D:\\Data\\SER\\Edited\\2026pilot\\Michael\\fish-dlc\\Calibration_Wand-Michael-2026-02-17\\dlc-models\\iteration-0\\Calibration_WandFeb17-trainset95shuffle1\\train\\snapshot', 'log_dir': 'log', 'global_scale': 0.25, 'location_refinement': True, 'locref_stdev': 7.2801, 'locref_loss_weight': 0.05, 'locref_huber_loss': True, 'optimizer': 'sgd', 'intermediate_supervision': False, 'intermediate_supervision_layer': 12, 'regularize': False, 'weight_decay': 0.0001, 'crop_pad': 0, 'scoremap_dir': 'test', 'batch_size': 1, 'dataset_type': 'imgaug', 'deterministic': False, 'mirror': False, 'pairwise_huber_loss': False, 'weigh_only_present_joints': False, 'partaffinityfield_pre

...

c:\Users\bhvlab\.conda\envs\dlc\lib\site-packages\tensorflow\python\keras\engine\base_layer_v1.py:1694: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  warnings.warn('`layer.apply` is deprecated and '


Loading ImageNet-pretrained resnet_50
Max_iters overwritten as 30000
Display_iters overwritten as 200
Save_iters overwritten as 5000
Training parameter:
{'stride': 8.0, 'weigh_part_predictions': False, 'weigh_negatives': False, 'fg_fraction': 0.25, 'mean_pixel': [123.68, 116.779, 103.939], 'shuffle': True, 'snapshot_prefix': 'D:\\Data\\SER\\Edited\\2026pilot\\Michael\\fish-dlc\\SandbarShark_Top-Michael-2026-02-17\\dlc-models\\iteration-0\\SandbarShark_TopFeb17-trainset95shuffle1\\train\\snapshot', 'log_dir': 'log', 'global_scale': 0.25, 'location_refinement': True, 'locref_stdev': 7.2801, 'locref_loss_weight': 0.05, 'locref_huber_loss': True, 'optimizer': 'sgd', 'intermediate_supervision': False, 'intermediate_supervision_layer': 12, 'regularize': False, 'weight_decay': 0.0001, 'crop_pad': 0, 'scoremap_dir': 'test', 'batch_size': 1, 'dataset_type': 'imgaug', 'deterministic': False, 'mirror': False, 'pairwise_huber_loss': False, 'weigh_only_present_joints': False, 'partaffinityfield_pre

iteration: 200 loss: 0.0377 lr: 0.005
iteration: 400 loss: 0.0071 lr: 0.005
iteration: 600 loss: 0.0051 lr: 0.005
iteration: 800 loss: 0.0040 lr: 0.005
iteration: 1000 loss: 0.0036 lr: 0.005
iteration: 1200 loss: 0.0034 lr: 0.005
iteration: 1400 loss: 0.0033 lr: 0.005
iteration: 1600 loss: 0.0028 lr: 0.005
iteration: 1800 loss: 0.0031 lr: 0.005
iteration: 2000 loss: 0.0029 lr: 0.005
iteration: 2200 loss: 0.0027 lr: 0.005
iteration: 2400 loss: 0.0027 lr: 0.005
iteration: 2600 loss: 0.0028 lr: 0.005
iteration: 2800 loss: 0.0027 lr: 0.005
iteration: 3000 loss: 0.0028 lr: 0.005
iteration: 3200 loss: 0.0027 lr: 0.005
iteration: 3400 loss: 0.0028 lr: 0.005
iteration: 3600 loss: 0.0025 lr: 0.005
iteration: 3800 loss: 0.0028 lr: 0.005
iteration: 4000 loss: 0.0026 lr: 0.005
iteration: 4200 loss: 0.0026 lr: 0.005
iteration: 4400 loss: 0.0025 lr: 0.005
iteration: 4600 loss: 0.0025 lr: 0.005
iteration: 4800 loss: 0.0024 lr: 0.005
iteration: 5000 loss: 0.0024 lr: 0.005
iteration: 5200 loss: 0.0025 

The network is now trained and ready to evaluate. Use the function 'evaluate_network' to evaluate the network.

✅ All Training Loops Completed.
Hibernating in 60 seconds, press ctrl+C to cancel.


0

## 🏋️ Step 4: Analysis

This step is where the trained model will generalize to the rest of the video you have provided. You labeled a fair amount of frames, it is going to use what you trained it to do and labele the 50,000 or so odd frames remaining to provide you with a neat output video, a csv file, and an h5 file with the video's tracking data. 

This one also takes a long time, and will make your computer hot. The timeframe is closer to an hour than overnight, but regardless, be prepared to step away from the computer and leave it to its task. 

In [ ]:
# --- STEP 4: PRE-PROCESSING (1080p DOWNSCALE) ---
# This block converts 4K videos and saves them as a 1080p copy 
# using the VIDEO_DIR selected in the Step 1 GUI.

import os
import subprocess
import glob

# Check if GUI variables exist
if 'VIDEO_DIR' not in globals():
    raise ValueError("VIDEO_DIR not found. Please run Step 1 (Project Setup) first.")

# Use the directory from the GUI
SOURCE_DIR = VIDEO_DIR 
OUTPUT_DIR = os.path.join(SOURCE_DIR, "1080p_Ready")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("🚀 Starting physical downscale to 1080p @ 30 fps...")
print(f"📂 Source: {SOURCE_DIR}")
print(f"📂 Output: {OUTPUT_DIR}\n")

# Find all mp4s in the source directory
videos = glob.glob(os.path.join(SOURCE_DIR, "*.mp4"))

for video_path in videos:
    filename = os.path.basename(video_path)
    
    # Skip if we are looking at the output directory itself
    if "1080p_Ready" in video_path: 
        continue
        
    output_path = os.path.join(OUTPUT_DIR, filename)
    
    # Skip if file already exists to save time
    if os.path.exists(output_path):
        print(f"   ⏩ Skipping {filename} (Already exists)")
        continue
    
    print(f"🎥 Processing: {filename}...")
    
    # FFmpeg Command:
    # -vf scale=1920:1080,fps=30 (Forces 1080p and 30fps)
    # -c:a copy (Copies audio untouched)
    command = [
        "ffmpeg", "-y", "-i", video_path,
        "-vf", "scale=1920:1080,fps=30",
        "-c:a", "copy", output_path
    ]
    
    try:
        subprocess.run(command, check=True, stderr=subprocess.DEVNULL)
        print(f"   ✅ Done.")
    except Exception as e:
        print(f"   ❌ Failed to convert {filename}: {e}")

print("\n🎉 Conversion Complete.")

🚀 Starting physical downscale to 1080p @ 30 fps...
📂 Output folder: D:\Data\SER\Edited\2026pilot\Michael\1080p_Ready

🎥 Processing: 20260121_SERt2tp_goproGX010004_trimmed.mp4 (This may take a few minutes)
   ✅ Done.
🎥 Processing: 20260121_SERt2w1_goproGX010125_trimmed.mp4 (This may take a few minutes)
   ✅ Done.
🎥 Processing: 20260121_SERt2w2_goproGX010153_trimmed.mp4 (This may take a few minutes)
   ✅ Done.
🎥 Processing: 20260121_SERt2w3_akaso131835_trimmed.mp4 (This may take a few minutes)
   ✅ Done.

🎉 Conversion Complete.


In [ ]:
# --- STEP 4: ANALYSIS & PREP (v7.4 - Script-Folder-Proof) ---
import os
import shutil
import pandas as pd
from pathlib import Path
import deeplabcut
import subprocess
import glob

# 1. DYNAMIC PATH SENSING
# Uses the VIDEO_DIR and WORKING_DIR defined in the Step 1 GUI
if 'VIDEO_DIR' not in globals() or 'WORKING_DIR' not in globals():
    raise ValueError("GUI Variables not found. Please run Step 1 (Project Setup) first.")

# Define where the downscaled videos live
OUTPUT_DIR = os.path.join(VIDEO_DIR, "1080p_Ready")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. PHYSICAL DOWNSCALE (FFMPEG)
print(f"🚀 Starting physical downscale to 1080p @ 30 fps...")
print(f"📂 Source: {VIDEO_DIR}")
print(f"📂 Output: {OUTPUT_DIR}\n")

videos = glob.glob(os.path.join(VIDEO_DIR, "*.mp4"))

for video_path in videos:
    filename = os.path.basename(video_path)
    # Skip if already in the output folder to avoid infinite loops
    if "1080p_Ready" in video_path: continue
    
    output_path = os.path.join(OUTPUT_DIR, filename)
    
    if os.path.exists(output_path):
        print(f"   ⏩ Skipping {filename} (Already exists)")
        continue

    print(f"🎥 Processing: {filename}...")
    command = [
        "ffmpeg", "-y", "-i", video_path,
        "-vf", "scale=1920:1080,fps=30",
        "-c:a", "copy", output_path
    ]
    
    try:
        subprocess.run(command, check=True, stderr=subprocess.DEVNULL)
        print(f"   ✅ Done.")
    except Exception as e:
        print(f"   ❌ Failed to convert {filename}: {e}")

# 3. AUTO-DETECT 1080p VIDEOS FOR ANALYSIS
print(f"\nScanning {OUTPUT_DIR} for analysis targets...")
ready_videos = [os.path.join(OUTPUT_DIR, f) for f in os.listdir(OUTPUT_DIR) if f.lower().endswith(".mp4")]
video_map = {}

for v in ready_videos:
    v_lower = os.path.basename(v).lower()
    if "top" in v_lower or "tp" in v_lower: video_map['Top'] = v
    elif "left" in v_lower or "w1" in v_lower: video_map['Left'] = v
    elif "right" in v_lower or "w2" in v_lower: video_map['Right'] = v
    elif "front" in v_lower or "w3" in v_lower: video_map['Front'] = v

print("Found Videos to Analyze:", video_map)

# 4. RUN DLC ANALYSIS
pose_folder = os.path.join(WORKING_DIR, "pose_2d")
os.makedirs(pose_folder, exist_ok=True)

# A. ANALYZE SHARKS
print("\n🦈 Analyzing Sharks...")
for cam_name, video_path in video_map.items():
    cfg_key = [k for k in project_configs.keys() if cam_name in k and 'Wand' not in k]
    if not cfg_key: continue
    cfg = project_configs[cfg_key[0]]
    print(f"   Processing {cam_name} View...")
    
    scorername = deeplabcut.analyze_videos(cfg, [video_path], save_as_csv=True)
    
    # Locate the generated .h5 file
    source_h5 = str(Path(video_path).with_suffix('')) + scorername + ".h5"
    dest_h5 = os.path.join(pose_folder, f"Trial1_Cam_{cam_name}.h5")
    
    if os.path.exists(source_h5):
        shutil.copy(source_h5, dest_h5)
        print(f"   ✅ Saved {cam_name} data to pose_2d.")
    else:
        print(f"   ❌ ERROR: DLC failed to generate {source_h5}.")

# B. ANALYZE WAND
print("\n🪄 Tracking Calibration Wand...")
wand_cfg = project_configs['Wand']

for cam_name, video_path in video_map.items():
    print(f"   Searching for wand in {cam_name}...")
    scorername = deeplabcut.analyze_videos(wand_cfg, [video_path], save_as_csv=True)
    
    source_csv = str(Path(video_path).with_suffix('')) + scorername + ".csv"
    
    if not os.path.exists(source_csv):
        print(f"   ❌ ERROR: Wand analysis failed to generate {source_csv}")
        continue
        
    print(f"   🧹 Filtering noise ({WAND_START_FRAME}-{WAND_END_FRAME})...")
    
    df = pd.read_csv(source_csv, header=[0,1,2], index_col=0)
    mask_bad = (df.index < WAND_START_FRAME) | (df.index > WAND_END_FRAME)
    scorer = df.columns.get_level_values(0)[0]
    for bp in df.columns.get_level_values(1).unique():
        df.loc[mask_bad, (scorer, bp, 'likelihood')] = 0
        
    dest_csv = os.path.join(pose_folder, f"Calibration_Cam_{cam_name}.csv")
    df.to_csv(dest_csv)

print(f"\n✅ Ready for Calibration.")

Scanning D:/Data/SER/Edited/2026pilot/Michael for MP4s...
Found Videos to Analyze: {'Top': 'D:/Data/SER/Edited/2026pilot/Michael\\20260121_SERt2tp_goproGX010004_trimmed.mp4', 'Left': 'D:/Data/SER/Edited/2026pilot/Michael\\20260121_SERt2w1_goproGX010125_trimmed.mp4', 'Right': 'D:/Data/SER/Edited/2026pilot/Michael\\20260121_SERt2w2_goproGX010153_trimmed.mp4', 'Front': 'D:/Data/SER/Edited/2026pilot/Michael\\20260121_SERt2w3_akaso131835_trimmed.mp4'}

🦈 Analyzing Sharks...
   Processing Top View...
Using snapshot-30000 for model D:/Data/SER/Edited/2026pilot/Michael/fish-dlc\SandbarShark_Top-Michael-2026-02-17\dlc-models\iteration-0\SandbarShark_TopFeb17-trainset95shuffle1


c:\Users\bhvlab\.conda\envs\dlc\lib\site-packages\tensorflow\python\keras\engine\base_layer_v1.py:1694: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  warnings.warn('`layer.apply` is deprecated and '


Starting to analyze %  D:/Data/SER/Edited/2026pilot/Michael\20260121_SERt2tp_goproGX010004_trimmed.mp4
Loading  D:/Data/SER/Edited/2026pilot/Michael\20260121_SERt2tp_goproGX010004_trimmed.mp4
Duration of video [s]:  301.07 , recorded with  30.0 fps!
Overall # of frames:  9032  found with (before cropping) frame dimensions:  1920 1080
Starting to extract posture


100%|██████████| 9032/9032 [12:01<00:00, 12.52it/s]


Saving results in D:\Data\SER\Edited\2026pilot\Michael...
Saving csv poses!
The videos are analyzed. Now your research can truly start! 
 You can create labeled videos with 'create_labeled_video'
If the tracking is not satisfactory for some videos, consider expanding the training set. You can use the function 'extract_outlier_frames' to extract a few representative outlier frames.
   ✅ Saved Top data.
   Processing Left View...
Using snapshot-30000 for model D:/Data/SER/Edited/2026pilot/Michael/fish-dlc\SandbarShark_Left-Michael-2026-02-17\dlc-models\iteration-0\SandbarShark_LeftFeb17-trainset95shuffle1


c:\Users\bhvlab\.conda\envs\dlc\lib\site-packages\tensorflow\python\keras\engine\base_layer_v1.py:1694: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  warnings.warn('`layer.apply` is deprecated and '


Starting to analyze %  D:/Data/SER/Edited/2026pilot/Michael\20260121_SERt2w1_goproGX010125_trimmed.mp4
Loading  D:/Data/SER/Edited/2026pilot/Michael\20260121_SERt2w1_goproGX010125_trimmed.mp4
Duration of video [s]:  309.77 , recorded with  30.0 fps!
Overall # of frames:  9293  found with (before cropping) frame dimensions:  1920 1080
Starting to extract posture


 24%|██▎       | 2184/9293 [02:54<09:29, 12.48it/s]
c:\Users\bhvlab\.conda\envs\dlc\lib\site-packages\tensorflow\python\keras\engine\base_layer_v1.py:1694: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  warnings.warn('`layer.apply` is deprecated and '


The videos are analyzed. Now your research can truly start! 
 You can create labeled videos with 'create_labeled_video'
If the tracking is not satisfactory for some videos, consider expanding the training set. You can use the function 'extract_outlier_frames' to extract a few representative outlier frames.
   ❌ ERROR: DLC failed to generate D:\Data\SER\Edited\2026pilot\Michael\20260121_SERt2w1_goproGX010125_trimmedDLC_resnet50_SandbarShark_LeftFeb17shuffle1_30000.h5.
   Processing Right View...
Using snapshot-30000 for model D:/Data/SER/Edited/2026pilot/Michael/fish-dlc\SandbarShark_Right-Michael-2026-02-17\dlc-models\iteration-0\SandbarShark_RightFeb17-trainset95shuffle1
Starting to analyze %  D:/Data/SER/Edited/2026pilot/Michael\20260121_SERt2w2_goproGX010153_trimmed.mp4
Loading  D:/Data/SER/Edited/2026pilot/Michael\20260121_SERt2w2_goproGX010153_trimmed.mp4
Duration of video [s]:  296.33 , recorded with  30.0 fps!
Overall # of frames:  8890  found with (before cropping) frame dimens

  1%|▏         | 128/8890 [00:10<11:32, 12.66it/s]

In [ ]:
# --- STEP 4.5: VERIFY TRACKING QUALITY ---
import deeplabcut
import os

print("🎬 Generating Labeled Videos for Verification...")

for cam_name, video_path in video_map.items():
    cfg_key = [k for k in project_configs.keys() if cam_name in k and 'Wand' not in k]
    if not cfg_key: continue
    
    cfg = project_configs[cfg_key[0]]
    print(f"   Rendering {cam_name}...")
    
    # This reads your newly generated .h5 file and draws the dots on the video
    deeplabcut.create_labeled_video(cfg, [video_path], draw_skeleton=True)

print("\n✅ Done! Open the 'Michael' folder and check the new videos ending in '_labeled.mp4'.")

## 🏋️ Step 5: Calibration

So far the process has been training and evaluating a separate 2D model for each camera. This step uses the calibration data from the wand and applies it to the data output by those 2D models to create a 3D dataset. 

In [ ]:
# --- STEP 5: RUN 3D CALIBRATION ---
import subprocess

# Ensure we are looking in the project root for the TOML, not the /Scripts folder
config_path = os.path.abspath(os.path.join(WORKING_DIR, "config.toml"))

print(f"📐 Running Anipose Calibration using: {config_path}")
subprocess.run(["anipose", "calibrate", "--config", config_path], check=True)

print("🧊 Triangulating 3D Points...")
subprocess.run(["anipose", "triangulate", "--config", config_path], check=True)